In [0]:
from pyspark.sql.functions import to_timestamp

sessions_data = [
    (1, 1, "2024-01-01 00:00:00"),
    (2, 2, "2024-01-02 00:00:00"),
    (3, 3, "2024-01-05 00:00:00"),
    (4, 3, "2024-01-05 00:00:00"),
    (5, 4, "2024-01-03 00:00:00"),
    (6, 4, "2024-01-03 00:00:00"),
    (7, 5, "2024-01-04 00:00:00"),
    (8, 5, "2024-01-04 00:00:00"),
    (9, 3, "2024-01-05 00:00:00"),
    (10, 5, "2024-01-04 00:00:00")
]

sessions_columns = ["session_id", "user_id", "session_date"]

sessions_df = (
    spark.createDataFrame(sessions_data, sessions_columns)
         .withColumn("session_date", to_timestamp("session_date"))
)

sessions_df.show()
sessions_df.printSchema()
order_data = [
    (1, 1, 152, "2024-01-01 00:00:00"),
    (2, 2, 485, "2024-01-02 00:00:00"),
    (3, 3, 398, "2024-01-05 00:00:00"),
    (4, 3, 320, "2024-01-05 00:00:00"),
    (5, 4, 156, "2024-01-03 00:00:00"),
    (6, 4, 121, "2024-01-03 00:00:00"),
    (7, 5, 238, "2024-01-04 00:00:00"),
    (8, 5, 70, "2024-01-04 00:00:00"),
    (9, 3, 152, "2024-01-05 00:00:00"),
    (10, 5, 171, "2024-01-04 00:00:00")
]

order_columns = ["order_id", "user_id", "order_value", "order_date"]

order_summary_df = (
    spark.createDataFrame(order_data, order_columns)
         .withColumn("order_date", to_timestamp("order_date"))
)

order_summary_df.show()
order_summary_df.printSchema()


In [0]:
from pyspark.sql.functions import col,countDistinct,sum
df=sessions_df.alias("t1").join(order_summary_df.alias("t2"),(col("t1.user_id") == col("t2.user_id")) &
        (col("t1.session_date") == col("t2.order_date")),how="inner")

df.groupBy("t1.user_id","order_date").agg(countDistinct("order_id").alias("distinct_ord"),sum("order_value").alias("order_value_total")).show()

In [0]:
from pyspark.sql.functions import to_timestamp

fact_events_data = [
    (1, "2020-02-28", "3668-QPYBK", "Sendit", "desktop", "message sent", 3),
    (2, "2020-02-28", "7892-POOKP", "Connectix", "mobile", "file received", 2),
    (3, "2020-04-03", "9763-GRSKD", "Zoomit", "desktop", "video call received", 7),
    (4, "2020-04-02", "9763-GRSKD", "Connectix", "desktop", "video call received", 7),
    (5, "2020-02-06", "9237-HQITU", "Sendit", "desktop", "video call received", 7),
    (6, "2020-02-27", "8191-XWSZG", "Connectix", "desktop", "file received", 2),
    (7, "2020-04-03", "9237-HQITU", "Connectix", "desktop", "video call received", 7),
    (8, "2020-03-01", "9237-HQITU", "Connectix", "mobile", "message received", 4),
    (9, "2020-04-02", "4190-MFLUW", "Connectix", "mobile", "video call received", 7),
    (10, "2020-04-21", "9763-GRSKD", "Sendit", "desktop", "file received", 2)
]

fact_events_columns = [
    "id",
    "time_id",
    "user_id",
    "customer_id",
    "client_id",
    "event_type",
    "event_id"
]

fact_events_df = (
    spark.createDataFrame(fact_events_data, fact_events_columns)
         .withColumn("time_id", to_timestamp("time_id"))
)

fact_events_df.show(truncate=False)
fact_events_df.printSchema()


In [0]:
from pyspark.sql.functions import date_format,countDistinct
fact_events_df1=fact_events_df.withColumn("month_txn",date_format("time_id","MMM"))
fact_events_df1.groupBy("client_id","month_txn").agg(countDistinct(col("user_id")).alias("count_user")).show()

In [0]:
ms_user_data = [
    (1, 101),
    (2, 102),
    (3, 103),
    (4, 104),
    (5, 105)
]

ms_user_columns = ["user_id", "acc_id"]

ms_user_dimension_df = spark.createDataFrame(
    ms_user_data, ms_user_columns
)

ms_user_dimension_df.show()
ms_user_dimension_df.printSchema()

ms_acc_data = [
    (101, "Yes"),
    (102, "No"),
    (103, "Yes"),
    (104, "No"),
    (105, "No")
]

ms_acc_columns = ["acc_id", "paying_customer"]

ms_acc_dimension_df = spark.createDataFrame(
    ms_acc_data, ms_acc_columns
)

ms_acc_dimension_df.show()
ms_acc_dimension_df.printSchema()

from pyspark.sql.functions import to_date

ms_download_data = [
    ("2024-10-01", 1, 10),
    ("2024-10-01", 2, 15),
    ("2024-10-02", 1, 8),
    ("2024-10-02", 3, 12),
    ("2024-10-02", 4, 20),
    ("2024-10-03", 2, 25),
    ("2024-10-03", 5, 18)
]

ms_download_columns = ["date", "user_id", "downloads"]

ms_download_facts_df = (
    spark.createDataFrame(ms_download_data, ms_download_columns)
         .withColumn("date", to_date("date"))
)

ms_download_facts_df.show()
ms_download_facts_df.printSchema()




In [0]:
from pyspark.sql.functions import col, sum, when, to_date

result_df = (
    ms_download_facts_df.alias("d")
    .join(ms_user_dimension_df.alias("u"), col("d.user_id") == col("u.user_id"))
    .join(ms_acc_dimension_df.alias("a"), col("u.acc_id") == col("a.acc_id"))
    .groupBy(to_date(col("d.date")).alias("date"))
    .agg(
        sum(
            when(col("a.paying_customer") == "No", col("d.downloads"))
            .otherwise(0)
        ).alias("non_paying_downloads"),
        sum(
            when(col("a.paying_customer") == "Yes", col("d.downloads"))
            .otherwise(0)
        ).alias("paying_downloads")
    )
    .filter(col("non_paying_downloads") > col("paying_downloads"))
    .orderBy("date")
)

result_df.show()
